# Clustering: Perfiles de divorcios
¿Se pueden identificar patrones de segmentación socioeconómica, geográfica y étnica en los divorcios guatemaltecos?

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/data_divorce.csv')

In [3]:
# Crear variables

# Educacion
def simplificar_educacion(nivel):
    if nivel in ['Ninguno', 'Primaria']:
        return 0  # Básica o menos
    elif nivel == 'Basico':
        return 1  # Secundaria básica
    elif nivel == 'Diversificado':
        return 2  # Secundaria completa
    elif nivel in ['Universitario', 'Postgrado']:
        return 3  # Superior
    else:
        return -1  # Ignorado
    
df['EDU_HOMBRE'] = df['ESCHOM'].apply(simplificar_educacion)
df['EDU_MUJER'] = df['ESCMUJ'].apply(simplificar_educacion)
df['EDU_MAX'] = df[['EDU_HOMBRE', 'EDU_MUJER']].max(axis=1)

# Contexto geográfico
deptos_urbanos = ['Guatemala', 'Escuintla', 'Sacatepequez', 'Quetzaltenango', 'Retalhuleu']
df['URBANO'] = df['DEPOCU'].isin(deptos_urbanos).astype(int)

# Pareja indigena - contexto cultural
def clasificar_etnia(etnia):
    if pd.isna(etnia) or etnia == 'Ignorado':
        return -1
    elif etnia in ['Indigena', 'Maya', 'Garifuna', 'Xinca']:
        return 1
    else:
        return 0

df['INDIGENA_HOMBRE'] = df['GETHOM'].apply(clasificar_etnia)
df['INDIGENA_MUJER'] = df['GETMUJ'].apply(clasificar_etnia)
df['PAREJA_INDIGENA'] = ((df['INDIGENA_HOMBRE'] == 1) | (df['INDIGENA_MUJER'] == 1)).astype(int)


In [4]:
# Crear dataset
features = ['EDU_MAX', 'URBANO', 'PAREJA_INDIGENA']
X = df[features].copy()

# Filtrar valores válidos
X = X[X['EDU_MAX'] >= 0]  # Eliminar educación ignorada
X = X[X.index.isin(df[(df['INDIGENA_HOMBRE'] >= 0) | (df['INDIGENA_MUJER'] >= 0)].index)]  # Eliminar etnia desconocida
X = X.dropna()

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Metodo del codo

Para saber cuantos clusters (k) usar

### Analisis de Silhoutte

- "> 0.5 → clustering fuerte"
- "0.25 – 0.5 → clustering razonable"
- "< 0.25 → estructura débil"

In [5]:
inercias = []
siluetas = []

K_range = range(2, 8) 
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inercias.append(kmeans.inertia_)
    
    # Calcular y guardar Silhouette
    score = silhouette_score(X_scaled, labels)
    siluetas.append(score)
    
    print(f"Para k={k}, Silhouette Score: {score:.4f}")

# Graficos comparativos
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Gráfica del Codo
ax1.plot(K_range, inercias, marker='o', color='b')
ax1.set_title('Método del Codo (Inercia)')
ax1.set_xlabel('Número de Clusters (K)')
ax1.set_ylabel('Inercia')

# Gráfica de Silhouette
ax2.plot(K_range, siluetas, marker='o', color='g')
ax2.set_title('Análisis de Silhouette')
ax2.set_xlabel('Número de Clusters (K)')
ax2.set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

Para k=2, Silhouette Score: 0.4513
Para k=3, Silhouette Score: 0.5729
Para k=4, Silhouette Score: 0.6246


KeyboardInterrupt: 

In [ ]:
k_optimo = 4
kmeans_final = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
X['cluster'] = kmeans_final.fit_predict(X_scaled)

# Agregar cluster al dataframe original
df_con_cluster = df.loc[X.index].copy()
df_con_cluster['cluster'] = X['cluster']

# Crear tabla resumen
resumen = df_con_cluster.groupby('cluster').agg({
    'EDU_MAX': 'mean',
    'URBANO': 'mean',
    'PAREJA_INDIGENA': 'mean'
}).round(2)

# Agregar tamaño de cada cluster
resumen['N_casos'] = df_con_cluster.groupby('cluster').size()
resumen['Porcentaje'] = (resumen['N_casos'] / len(df_con_cluster) * 100).round(1)

# Renombrar columnas para claridad
resumen.columns = ['Edu_Max_Promedio', 'Urbano_%', 'Indigena_%', 'N_casos', 'Porcentaje_%']
resumen['Urbano_%'] = (resumen['Urbano_%'] * 100).round(1)
resumen['Indigena_%'] = (resumen['Indigena_%'] * 100).round(1)

resumen

,Edu_Max_Promedio,Urbano_%,Indigena_%,N_casos,Porcentaje_%
cluster,,,,,
0,1.03,28.0,100.0,6193,17.4
1,0.31,0.0,0.0,5445,15.3
2,1.91,100.0,0.0,16033,45.1
3,2.20,0.0,0.0,7918,22.2


In [ ]:
# Interpretacion del cluster

for i in range(k_optimo):
    cluster_data = df_con_cluster[df_con_cluster['cluster'] == i]
    
    print(f"\nCLUSTER {i}")
    print('-'*50)
    print(f"Tamaño: {len(cluster_data):,} casos ({len(cluster_data)/len(df_con_cluster)*100:.1f}%)")
    
    print(f"\nEDUCACIÓN:")
    print(f"Nivel máximo promedio: {cluster_data['EDU_MAX'].mean():.2f}")
    print(f"Distribución:")
    edu_dist = cluster_data['EDU_MAX'].value_counts().sort_index()
    for nivel, count in edu_dist.items():
        niveles = {0: 'Básica', 1: 'Secundaria', 2: 'Diversificado', 3: 'Superior'}
        print(f"{niveles.get(nivel, nivel)}: {count:,} ({count/len(cluster_data)*100:.1f}%)")
    
    print(f"\nCONTEXTO:")
    print(f"Urbano: {cluster_data['URBANO'].mean()*100:.1f}%")
    print(f"Rural: {(1-cluster_data['URBANO'].mean())*100:.1f}%")
    
    print(f"\nETNIA:")
    print(f"Pareja con componente indígena: {cluster_data['PAREJA_INDIGENA'].mean()*100:.1f}%")
    print(f"Pareja sin componente indígena: {(1-cluster_data['PAREJA_INDIGENA'].mean())*100:.1f}%")
    
    print(f"\nDEPARTAMENTO PRINCIPAL:")
    depto_mode = cluster_data['DEPOCU'].mode()
    if len(depto_mode) > 0:
        print(f"{depto_mode[0]}")


CLUSTER 0
--------------------------------------------------
Tamaño: 6,193 casos (17.4%)

EDUCACIÓN:
Nivel máximo promedio: 1.03
Distribución:
Básica: 2,702 (43.6%)
Secundaria: 1,008 (16.3%)
Diversificado: 2,066 (33.4%)
Superior: 417 (6.7%)

CONTEXTO:
Urbano: 27.9%
Rural: 72.1%

ETNIA:
Pareja con componente indígena: 100.0%
Pareja sin componente indígena: 0.0%

DEPARTAMENTO PRINCIPAL:
Quetzaltenango

CLUSTER 1
--------------------------------------------------
Tamaño: 5,445 casos (15.3%)

EDUCACIÓN:
Nivel máximo promedio: 0.31
Distribución:
Básica: 3,746 (68.8%)
Secundaria: 1,699 (31.2%)

CONTEXTO:
Urbano: 0.0%
Rural: 100.0%

ETNIA:
Pareja con componente indígena: 0.0%
Pareja sin componente indígena: 100.0%

DEPARTAMENTO PRINCIPAL:
San Marcos

CLUSTER 2
--------------------------------------------------
Tamaño: 16,033 casos (45.1%)

EDUCACIÓN:
Nivel máximo promedio: 1.91
Distribución:
Básica: 1,985 (12.4%)
Secundaria: 1,687 (10.5%)
Diversificado: 8,213 (51.2%)
Superior: 4,148 (25.9%)
